# 🧬 GIADA — Task19 originale: Ih e Im
24 modelli indipendenti: due meccanismi a un gate, due obiettivi, due capacità, tre seed. Audit nativo, LUT, stato estremo, rollout1000 e latenzaGPU. Nessun Dataset da montare.


In [ ]:
import os
os.environ['GIADA_CODE_REVISION'] = '9f5d49fb34dfd077d8fc0ec5027e3af3c179900c'
os.environ['GIADA_MANUAL_DOWNLOAD'] = '0'


In [ ]:
from pathlib import Path
import base64,json,os,shutil,subprocess,sys,uuid
from IPython.display import display,Javascript
WORK=Path('/kaggle/working')/('giada_task19_'+uuid.uuid4().hex[:8]);WORK.mkdir()
REPO=WORK/'giada';TEACHER=WORK/'teacher'
subprocess.run(['git','clone','https://github.com/Zagred47/giada.git',str(REPO)],check=True)
subprocess.run(['git','-C',str(REPO),'fetch','origin',os.environ.get('GIADA_CODE_REVISION','codex/surrogate-validity-audit')],check=True)
subprocess.run(['git','-C',str(REPO),'checkout','--detach','FETCH_HEAD'],check=True)
subprocess.run(['git','clone','https://github.com/SelfishGene/neuron_as_deep_net.git',str(TEACHER)],check=True)
subprocess.run(['git','-C',str(TEACHER),'checkout','--detach','074c4666300a8ad246601dab179a97a6942f0f29'],check=True)
REVISION=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
subprocess.run([sys.executable,'-m','pip','install','--quiet','neuron==8.2.7','numpy'],check=True)
os.environ['PATH']=str(Path(sys.executable).parent)+os.pathsep+os.environ.get('PATH','')
subprocess.run([sys.executable,'-c','import torch; assert torch.cuda.is_available(),"Attiva GPU CUDA"; print(torch.cuda.get_device_name(0))'],check=True)
print({'revision':REVISION})

## ⚗️ Contratto preregistrato
V e parametri canonici costanti; input V,x,dt. Stream appaiati e optimizer indipendenti; checkpoint0/1k/5k/15k/30k. Selezione solo development, comune ai3seed. Test fresh dopo freeze; OOD separato. La Task20 richiede entrambi i canali e tutti i seed entro le soglie registrate, inclusi i rollout.


In [ ]:
OUTPUT=Path('/kaggle/working/artifacts')/('giada_task19_single_gate_'+REVISION[:7]+'_'+uuid.uuid4().hex[:8])
completed=subprocess.run([sys.executable,'-u',str(REPO/'scripts/run_roadmap_task19.py'),'--teacher',str(TEACHER),'--output',str(OUTPUT)],check=False)
path=OUTPUT/('failure_report.json' if (OUTPUT/'failure_report.json').exists() else 'final_report.json')
report=json.loads(path.read_text()) if path.is_file() else {'valid':False,'error':f'worker interrupted: {completed.returncode}'}
display({k:report.get(k) for k in ('valid','single_gate_transfer_passed','per_channel_transfer','task20_authorized','fresh_used_for_selection','training_and_evaluation_seconds','error')})
print({'output':str(OUTPUT),'exit':completed.returncode})

In [ ]:
archive=Path(shutil.make_archive(str(Path('/kaggle/working')/OUTPUT.name),'zip',OUTPUT))
if os.environ.get('GIADA_MANUAL_DOWNLOAD','1')!='0':
 payload=base64.b64encode(archive.read_bytes()).decode('ascii')
 display(Javascript(f"const b=atob('{payload}');const a=new Uint8Array(b.length);for(let i=0;i<b.length;i++)a[i]=b.charCodeAt(i);const u=URL.createObjectURL(new Blob([a],{{type:'application/zip'}}));const l=document.createElement('a');l.href=u;l.download='{archive.name}';document.body.appendChild(l);l.click();l.remove();setTimeout(()=>URL.revokeObjectURL(u),1000);"))
print({'archive':str(archive),'size_mib':round(archive.stat().st_size/2**20,2)})